# Malware File Analyzer with Function Calling

**Author:** Andrea Agosti — [@cifvts](https://github.com/cifvts) — independent contributor

Check whether a file is malicious **without uploading it**: hash the file (SHA-256), query online threat-intelligence databases (VirusTotal, MalwareBazaar, AlienVault OTX), then let a Mistral model use **function calling** to run extra lookups and synthesize a plain-language verdict.

## What you'll learn
- How to expose external API lookups as tools to a Mistral model
- Running a tool-calling loop (model requests data, you execute, model concludes)
- Production patterns: per-source failure isolation, hash-only lookups (no raw file upload), graceful degradation when keys are missing

## Setup
Free API keys (all read from environment variables / Colab secrets — never hardcoded):
- `MISTRAL_API_KEY` — [console.mistral.ai](https://console.mistral.ai/api-keys/)
- `VT_API_KEY` — [VirusTotal](https://www.virustotal.com/gui/my-apikey) (free tier)
- `MBAZAAR_AUTH_KEY` — [bazaar.abuse.ch](https://bazaar.abuse.ch/) (free)
- `OTX_API_KEY` — optional, [AlienVault OTX](https://otx.alienvault.com/)



In [ ]:
# Pinned versions for reproducibility
!pip install -q "mistralai==3.10.0" "requests==2.32.3"



In [ ]:
import getpass, os

for var in ("MISTRAL_API_KEY", "VT_API_KEY", "MBAZAAR_AUTH_KEY"):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")
# OTX_API_KEY stays optional



## 1. Hash the file

SHA-256 lets us query databases by hash — the raw file never leaves your machine (only VirusTotal needs an upload for *unknown* files, which we skip here).



In [ ]:
import hashlib

def sha256_of(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

# Use the EICAR anti-malware test file (harmless industry-standard test string)
EICAR = 'X5O!P%@AP[4\\PZX54(P^)7CC)7}$EICAR-STANDARD-ANTIVIRUS-TEST-FILE!$H+H*'
with open("/content/eicar_test.com", "w") as f:
    f.write(EICAR)

file_hash = sha256_of("/content/eicar_test.com")
print("SHA-256:", file_hash)
assert file_hash == "275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f"



## 2. Resilient HTTP helper

Real APIs rate-limit and fail. This wrapper retries connection errors, timeouts, 429s and 5xx with exponential backoff (honoring `Retry-After`), and a safe JSON parser turns malformed responses into error entries instead of crashes.



In [ ]:
import time
import requests

TIMEOUT = 30
MAX_RETRIES = 3
BACKOFF_BASE = 2  # seconds; attempt n waits BACKOFF_BASE ** n

def request_with_retry(method: str, url: str, **kwargs) -> requests.Response:
    kwargs.setdefault("timeout", TIMEOUT)
    last_exc = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            r = requests.request(method, url, **kwargs)
            if (r.status_code == 429 or r.status_code >= 500) and attempt < MAX_RETRIES:
                time.sleep(float(r.headers.get("Retry-After", BACKOFF_BASE ** attempt)))
                continue
            return r
        except (requests.ConnectionError, requests.Timeout) as e:
            last_exc = e
            if attempt < MAX_RETRIES:
                time.sleep(BACKOFF_BASE ** attempt)
    raise last_exc

def safe_json(r: requests.Response) -> dict:
    try:
        return r.json()
    except ValueError:
        return {"error": f"non-JSON response (HTTP {r.status_code})"}



## 3. Threat-intel lookups

Three independent sources, each isolated so one failure doesn't kill the run.



In [ ]:
def virustotal_lookup(file_hash: str) -> dict:
    """VirusTotal v3 file report by hash."""
    r = request_with_retry(
        "GET",
        f"https://www.virustotal.com/api/v3/files/{file_hash}",
        headers={"x-apikey": os.environ["VT_API_KEY"]},
    )
    if r.status_code == 404:
        return {"source": "virustotal", "known": False}
    if r.status_code != 200:
        return {"source": "virustotal", "error": f"HTTP {r.status_code}"}
    data = safe_json(r)
    try:
        stats = data["data"]["attributes"]["last_analysis_stats"]
    except (KeyError, TypeError):
        return {"source": "virustotal", "error": data.get("error", "unexpected response shape")}
    return {"source": "virustotal", "known": True,
            "malicious": stats.get("malicious", 0),
            "suspicious": stats.get("suspicious", 0),
            "harmless": stats.get("harmless", 0),
            "undetected": stats.get("undetected", 0)}

def malwarebazaar_lookup(file_hash: str) -> dict:
    """abuse.ch MalwareBazaar (endpoint mb-api.abuse.ch, free Auth-Key)."""
    r = request_with_retry(
        "POST",
        "https://mb-api.abuse.ch/api/v1/",
        data={"query": "get_info", "hash": file_hash},
        headers={"Auth-Key": os.environ["MBAZAAR_AUTH_KEY"]},
    )
    if r.status_code != 200:
        return {"source": "malwarebazaar", "error": f"HTTP {r.status_code}"}
    data = safe_json(r)
    if data.get("error") == "Unauthorized":
        return {"source": "malwarebazaar", "error": "Auth-Key rejected"}
    status = data.get("query_status")
    if status == "file_not_found":
        return {"source": "malwarebazaar", "known": False}
    if status != "ok":
        return {"source": "malwarebazaar", "error": f"API status: {status}"}
    try:
        entry = data["data"][0]
    except (KeyError, IndexError, TypeError):
        return {"source": "malwarebazaar", "error": "unexpected response shape"}
    return {"source": "malwarebazaar", "known": True,
            "signature": entry.get("signature"),
            "file_type": entry.get("file_type"),
            "tags": entry.get("tags", [])}

def otx_lookup(file_hash: str) -> dict:
    """AlienVault OTX (optional key)."""
    if not os.environ.get("OTX_API_KEY"):
        return {"source": "otx", "skipped": "OTX_API_KEY not set (optional)"}
    r = request_with_retry(
        "GET",
        f"https://otx.alienvault.com/api/v1/indicators/file/{file_hash}/general",
        headers={"X-OTX-API-KEY": os.environ["OTX_API_KEY"]},
    )
    if r.status_code != 200:
        return {"source": "otx", "error": f"HTTP {r.status_code}"}
    data = safe_json(r)
    return {"source": "otx",
            "pulse_count": data.get("pulse_info", {}).get("count", 0)}

LOOKUPS = (virustotal_lookup, malwarebazaar_lookup, otx_lookup)



## 4. Aggregate the report



In [ ]:
import json
from dataclasses import dataclass, field

@dataclass
class AnalysisReport:
    file_hash: str
    results: list = field(default_factory=list)
    def to_dict(self) -> dict:
        return {"sha256": self.file_hash, "sources": self.results}

def build_report(file_hash: str) -> AnalysisReport:
    """Query every source for one hash; per-source failure isolation."""
    report = AnalysisReport(file_hash=file_hash)
    for lookup in LOOKUPS:
        try:
            report.results.append(lookup(file_hash))
        except Exception as e:  # one DB failing shouldn't kill the run
            report.results.append({"source": lookup.__name__, "error": str(e)})
    return report

report = build_report(file_hash)
print(json.dumps(report.to_dict(), indent=2))



## 5. Mistral verdict via a tool-calling loop

The model receives the report and may call the lookup tools itself for fresh data before giving a final verdict (MALICIOUS / SUSPICIOUS / CLEAN).



In [ ]:
from mistralai.client import Mistral

TOOLS = [
    {"type": "function",
     "function": {"name": "virustotal_lookup",
                  "description": "Look up a SHA-256 file hash on VirusTotal.",
                  "parameters": {"type": "object",
                                 "properties": {"file_hash": {"type": "string"}},
                                 "required": ["file_hash"]}}},
    {"type": "function",
     "function": {"name": "malwarebazaar_lookup",
                  "description": "Look up a SHA-256 file hash on abuse.ch MalwareBazaar.",
                  "parameters": {"type": "object",
                                 "properties": {"file_hash": {"type": "string"}},
                                 "required": ["file_hash"]}}},
]

SYSTEM_PROMPT = """You are a security analyst. Given file analysis results from
threat-intelligence databases, produce a verdict: MALICIOUS, SUSPICIOUS, or CLEAN.
Explain the evidence, cite detection counts and signatures, and note uncertainty.
Use the lookup tools if you need fresh data."""

LOOKUP_FUNCS = {"virustotal_lookup": virustotal_lookup,
                "malwarebazaar_lookup": malwarebazaar_lookup}

def mistral_verdict(report: AnalysisReport, max_rounds: int = 5) -> str:
    client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": json.dumps(report.to_dict(), indent=2)},
    ]
    for _ in range(max_rounds):
        try:
            response = client.chat.complete(model="mistral-small-latest",
                                            messages=messages, tools=TOOLS)
        except Exception as e:  # bad key, quota, outage — don't kill the run
            return f"SKIPPED: Mistral API call failed ({type(e).__name__}: {e}). The raw report above is still valid."
        msg = response.choices[0].message
        messages.append(msg)
        tool_calls = getattr(msg, "tool_calls", None)
        if not tool_calls:
            return msg.content or ""
        for call in tool_calls:
            func = LOOKUP_FUNCS.get(call.function.name)
            args = json.loads(call.function.arguments or "{}")
            result = func(**args) if func else {"error": "unknown tool"}
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "name": call.function.name,
                             "content": json.dumps(result)})
    return "Verdict not reached: exceeded max tool-calling rounds."

print(mistral_verdict(report))



## 6. Batch mode: ask about more hashes

No file on disk needed — analyze any number of hashes in one go. Each hash gets its own failure-isolated report:



In [ ]:
hashes_to_check = [
    "275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f",  # EICAR test file
    "44d88612fea8a8f36de82e1278abb02f",                              # EICAR MD5 (VT also accepts it)
    "d41d8cd98f00b204e9800998ecf8427e",                              # empty file MD5
]

reports = [build_report(h) for h in hashes_to_check]
for r in reports:
    verdict_line = next((s for s in r.results if s.get("known")), None)
    print(r.file_hash[:16], "->", "seen by a source" if verdict_line else "not found / errored")
print()
print(json.dumps([r.to_dict() for r in reports], indent=2)[:1500])



## Takeaways
- Function calling turns the model into an analyst that can pull fresh evidence itself.
- Hash-first lookups avoid uploading raw files; each source is failure-isolated, with retries/backoff for transient API failures.
- Batch mode scales the same pipeline to many hashes at once.
- Keys live in environment variables / Colab secrets, never in the notebook.

